# 03 — Model Training & Hyperparameter Tuning

**职责**: 加载原始数据 → 时序划分 → 训练集特征选择 → 全模型调参 → TimeSeriesSplit 交叉验证 → 测试集评估 → 保存模型

**修复的问题**:
1. ✅ 特征选择仅在训练集上进行，避免数据泄露
2. ✅ 所有模型均进行超参数调优
3. ✅ 使用 `TimeSeriesSplit` 替代普通 K-Fold
4. ✅ 预留 20% 数据作为独立测试集

**输入**: `data/raw/train_data.csv`（原始数据）  
**输出**: `outputs/models/*.pkl`

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

from config import DATA_RAW, TARGET, MODEL_DIR, RANDOM_STATE
from src.preprocessing import load_and_clean, build_feature_matrix
from src.feature_engineering import compute_ensemble_importance
from src.models import discover_models, save_model
from src.evaluate import cross_validate_all, results_to_dataframe

## 1. 加载原始数据 & 时序划分

从原始数据重新开始，按时间排序后按 80/20 划分训练集和测试集。  
使用**时序划分**（前 80% 训练、后 20% 测试），确保不会用"未来数据"训练。

In [3]:
df = load_and_clean(DATA_RAW)
df = df.sort_values('startdate').reset_index(drop=True)

X_full, y_full, all_features = build_feature_matrix(df, TARGET)
print(f"Full dataset: X={X_full.shape}, y={y_full.shape}")
print(f"Total features (before selection): {len(all_features)}")

split_idx = int(len(X_full) * 0.8)
X_train_raw, X_test_raw = X_full.iloc[:split_idx], X_full.iloc[split_idx:]
y_train, y_test = y_full.iloc[:split_idx], y_full.iloc[split_idx:]

print(f"\nTrain: {X_train_raw.shape}  (rows 0–{split_idx - 1})")
print(f"Test:  {X_test_raw.shape}  (rows {split_idx}–{len(X_full) - 1})")

/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/src/preprocessing.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['startdate'] = pd.to_datetime(df['startdate'])
/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/src/preprocessing.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['year']   = df['startdate'].dt.year
/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/src/preprocessing.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u

Full dataset: X=(375734, 242), y=(375734,)
Total features (before selection): 242

Train: (300587, 242)  (rows 0–300586)
Test:  (75147, 242)  (rows 300587–375733)


## 2. 特征选择（仅在训练集上）

使用 Lasso + ElasticNet + RandomForest 集成方法，**仅在训练集上**计算特征重要性。  
测试集完全不参与特征选择，杜绝数据泄露。

In [4]:
selected_score, full_score = compute_ensemble_importance(
    X_train_raw, y_train, top_n=30, use_cache=False
)
selected_features = selected_score.index.tolist()

print(f"\nSelected {len(selected_features)} features (train-only):")
print(selected_features)

X_train = X_train_raw[selected_features]
X_test = X_test_raw[selected_features]
print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")

  [1/3] Lasso …
  [2/3] ElasticNet …
  [3/3] RandomForest …
  Top-30 投票 ≥2/3: 25 个特征 (3/3=5, 2/3=20)
  Cached scores to /Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/outputs/cache/

Selected 25 features (train-only):
['contest-slp-14d__slp', 'contest-wind-h500-14d__wind-hgt-500', 'nmme-tmp2m-34w__gfdlflorb', 'nmme-tmp2m-34w__cfsv2', 'nmme-tmp2m-56w__cfsv2', 'nmme-tmp2m-56w__gfdlflorb', 'contest-wind-h850-14d__wind-hgt-850', 'nmme-tmp2m-56w__nasa', 'nmme-tmp2m-56w__gfdlflora', 'nmme-tmp2m-34w__nasa', 'elevation__elevation', 'nmme-tmp2m-34w__gfdlflora', 'contest-wind-h100-14d__wind-hgt-100', 'lon', 'contest-pres-sfc-gauss-14d__pres', 'contest-rhum-sig995-14d__rhum', 'contest-wind-vwnd-925-14d__wind-vwnd-925', 'wind-vwnd-925-2010-8', 'wind-hgt-10-2010-9', 'contest-wind-uwnd-925-14d__wind-uwnd-925', 'icec-2010-8', 'wind-hgt-10-2010-7', 'sst-2010-5', 'wind-uwnd-250-2010-10', 'wind-uwnd-250-2010-18']

X_train: (300587, 25), X_test: (75147, 25)


## 3. 超参数调优（所有模型）

对所有模型做 `RandomizedSearchCV`，使用 `TimeSeriesSplit` 作为交叉验证策略。  
**优化策略**：用训练集最近 30% 的数据做调参（时序上最接近测试集，更有代表性），3 折 CV，找到最优参数后再用全量训练集训练。

In [5]:
registry = discover_models()
print(f"Discovered {len(registry)} models: {list(registry.keys())}")

TUNE_FRAC = 0.3
tune_size = int(len(X_train) * TUNE_FRAC)
X_tune = X_train.iloc[-tune_size:]
y_tune = y_train.iloc[-tune_size:]
print(f"Tuning subset: {X_tune.shape} (last {TUNE_FRAC:.0%} of train)\n")

tscv_tune = TimeSeriesSplit(n_splits=3)
tuned_pipelines = {}

for name, entry in registry.items():
    grid = entry.param_grid()
    if not grid:
        print(f"[{name}] No param_grid, keeping default.\n")
        tuned_pipelines[name] = entry.build_pipeline()
        continue

    print(f"[{name}] Tuning ({len(grid)} param groups, n_iter=10)...")
    search = RandomizedSearchCV(
        entry.build_pipeline(), grid,
        n_iter=10, cv=tscv_tune,
        scoring='neg_mean_squared_error',
        n_jobs=-1, random_state=RANDOM_STATE, verbose=0,
    )
    search.fit(X_tune, y_tune)
    rmse = np.sqrt(-search.best_score_)
    print(f"  Best RMSE: {rmse:.3f}")
    print(f"  Best params: {search.best_params_}\n")
    tuned_pipelines[name] = search.best_estimator_

print(f"Done. Tuned {len(tuned_pipelines)} models.")

Discovered 6 models: ['catboost', 'elasticnet', 'lasso', 'lightgbm', 'random_forest', 'xgboost']
Tuning subset: (90176, 25) (last 30% of train)

[catboost] Tuning (3 param groups, n_iter=10)...
  Best RMSE: 2.682
  Best params: {'model__learning_rate': 0.2, 'model__iterations': 800, 'model__depth': 4}

[elasticnet] Tuning (2 param groups, n_iter=10)...
  Best RMSE: 2.853
  Best params: {'model__l1_ratio': 0.2, 'model__alpha': 0.5}

[lasso] Tuning (1 param groups, n_iter=10)...


/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


  Best RMSE: 2.795
  Best params: {'model__alpha': 0.5}

[lightgbm] Tuning (3 param groups, n_iter=10)...


/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/zhangjiay

  Best RMSE: 2.634
  Best params: {'model__n_estimators': 300, 'model__max_depth': 8, 'model__learning_rate': 0.2}

[random_forest] Tuning (3 param groups, n_iter=10)...
  Best RMSE: 3.076
  Best params: {'model__n_estimators': 100, 'model__min_samples_leaf': 5, 'model__max_depth': 15}

[xgboost] Tuning (3 param groups, n_iter=10)...
  Best RMSE: 2.648
  Best params: {'model__n_estimators': 800, 'model__max_depth': 4, 'model__learning_rate': 0.1}

Done. Tuned 6 models.


## 4. 测试集最终评估

用调参后的最优参数，在**全量训练集**上重新训练，然后在 **20% 测试集**上评估真实泛化能力。

In [6]:
test_rows = []
for name, pipe in tuned_pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    test_rows.append({'Model': name, 'Test RMSE': round(rmse, 3), 'Test R²': round(r2, 3)})
    print(f"[{name}] RMSE={rmse:.3f}, R²={r2:.3f}")

test_df = pd.DataFrame(test_rows).set_index('Model').sort_values('Test RMSE')
print(f"\nBest model on test set: {test_df.index[0]}")
test_df

[catboost] RMSE=1.903, R²=0.910
[elasticnet] RMSE=1.945, R²=0.906
[lasso] RMSE=1.941, R²=0.907


/Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[lightgbm] RMSE=1.993, R²=0.902
[random_forest] RMSE=1.950, R²=0.906
[xgboost] RMSE=1.958, R²=0.905

Best model on test set: catboost


,Test RMSE,Test R²
Model,,
catboost,1.903,0.910
lasso,1.941,0.907
elasticnet,1.945,0.906
random_forest,1.950,0.906
xgboost,1.958,0.905
lightgbm,1.993,0.902


## 5. 保存模型

保存所有调参后的模型（已在全量训练集上 fit），以及选中的特征列表。

In [7]:
import json

MODEL_DIR.mkdir(parents=True, exist_ok=True)

for name, pipe in tuned_pipelines.items():
    save_model(pipe, name)
    print(f"Saved: {name}")

features_path = MODEL_DIR / "selected_features.json"
with open(features_path, 'w') as f:
    json.dump(selected_features, f, indent=2)
print(f"\nSaved selected features → {features_path}")

Saved: catboost
Saved: elasticnet
Saved: lasso
Saved: lightgbm
Saved: random_forest
Saved: xgboost

Saved selected features → /Users/zhangjiayu/PycharmProjects/DS_Project/weather-forecast/outputs/models/selected_features.json
